# 04 · The same questions in SQL, at warehouse scale

**Produces:** the clean tables loaded into a database, and the outputs of `sql/03_profile.sql`, `sql/04_transformations.sql`, `sql/05_performance.sql`.

**Needs:** notebook 03 run; either a PostgreSQL connection or nothing (the notebook falls back to SQLite from the standard library).

Why SQL after pandas: the warehouse is where finance will re-run your numbers, the logical order of a query (FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY) is the interview question that never goes away, and window functions answer "compared with the previous month, per city" in one pass where pandas needs three.

## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

import sqlite3, os
PG_DSN = os.environ.get("KALPA_PG")          # e.g. "host=localhost dbname=kalpa user=postgres"; leave unset for SQLite
DB_PATH = CONFIG["clean"] / "kalpa.sqlite"
SQL_DIR = ROOT / "sql"

## 1 · DO: load the clean tables

In [ ]:
tables = {
    "bookings": pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["created_at", "slot_at", "collected_at"], dtype={"rescheduled_from": "string", "status_at_export": "string"}),
    "patients": pd.read_csv(CONFIG["clean"] / "patients_clean.csv.gz", dtype="string"),
    "invoices": pd.read_csv(CONFIG["clean"] / "invoices_clean.csv.gz", parse_dates=["invoice_date"]),
    "payments": pd.read_csv(CONFIG["clean"] / "payments_clean.csv.gz", parse_dates=["created_at"]),
    "plan": pd.read_csv(CONFIG["raw"] / "plan" / "plan_2026_H1.csv").melt(id_vars=["city", "metric"], var_name="month", value_name="target_tests"),
}
tables["plan"]["city"] = tables["plan"]["city"].replace({"Bangalore": "Bengaluru"})
if PG_DSN:
    import psycopg2
    con = psycopg2.connect(PG_DSN)
    print("PostgreSQL:", con.server_version)
else:
    if DB_PATH.exists(): DB_PATH.unlink()
    con = sqlite3.connect(DB_PATH)
    print("SQLite at", DB_PATH)
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)
    print(f"{name}: {len(df):,} rows")

## 2 · DO: run the SQL files

The files in `sql/` are written for PostgreSQL. `03_profile.sql` and `04_transformations.sql` also run on SQLite with the small substitutions `run_sql` makes; `05_performance.sql` (indexes and EXPLAIN) is PostgreSQL only and is skipped on SQLite.

In [ ]:
def run_sql(path, con):
    text = pathlib.Path(path).read_text()
    is_sqlite = isinstance(con, sqlite3.Connection)
    outputs = []
    for raw in text.split(";\n"):
        lines = [ln for ln in raw.strip().splitlines() if ln.strip()]
        title = next((ln.lstrip("- ").strip() for ln in lines if ln.strip().startswith("--")), "")[:90]
        stmt = "\n".join(ln for ln in lines if not ln.strip().startswith("--") and not ln.strip().startswith("\\"))
        if not stmt.strip():
            continue
        if is_sqlite:
            stmt = (stmt.replace("date_trunc('month', ", "strftime('%Y-%m', ").replace("::date", "").replace("::numeric", "")
                        .replace("EXTRACT(YEAR FROM ", "strftime('%Y', "))
            if "EXPLAIN" in stmt.upper() or "CREATE INDEX" in stmt.upper() or stmt.upper().startswith("ANALYZE"):
                continue
        try:
            outputs.append((title or stmt.splitlines()[0][:70], pd.read_sql_query(stmt, con)))
        except Exception as err:
            outputs.append((title or stmt.splitlines()[0][:70], pd.DataFrame({"error": [str(err)[:200]]})))
    return outputs

for title, df in run_sql(SQL_DIR / "03_profile.sql", con):
    print("\n==", title); print(df.head(12).to_string(index=False))

In [ ]:
t0 = time.time()
results_04 = run_sql(SQL_DIR / "04_transformations.sql", con)
sql_seconds = time.time() - t0
for title, df in results_04:
    print("\n==", title); print(df.head(12).to_string(index=False))
print(f"\n04_transformations.sql ran in {sql_seconds:.1f}s")

> **Interview angle.** Write the logical order in which a SELECT with a GROUP BY and a HAVING is evaluated, and use it to explain why a column alias from SELECT cannot be used in WHERE.
>
> *What a good answer covers:* FROM and JOIN, then WHERE, then GROUP BY, then HAVING, then SELECT, then ORDER BY and LIMIT. WHERE runs before SELECT has produced the alias, so the alias does not exist yet; HAVING runs after grouping so it can see aggregates; ORDER BY runs after SELECT so it can see aliases.

> **Interview angle.** When does a window function beat a GROUP BY, and what defines its result?
>
> *What a good answer covers:* When you need every row kept and a value computed over its neighbours: rank within a city, the previous month's value, a running total. The partition and the order inside the window define everything; a running total is only deterministic when the order has a tiebreaker.

## 3 · DO: performance, measured not guessed (PostgreSQL only)

In [ ]:
if PG_DSN:
    for title, df in run_sql(SQL_DIR / "05_performance.sql", con):
        print("\n==", title); print(df.head(20).to_string(index=False))
else:
    print("Skipped on SQLite. On PostgreSQL this file times a monthly rollup before and after an index on (city, slot_at) and prints EXPLAIN ANALYZE for both.")

## 4 · SUM

The same numbers you computed in pandas now come out of the database. Where they differ, one of the two is wrong; find out which before you go further. Write the SQL that finance will re-run into `sql/06_<your_brief>.sql` and reference it from the README.

## Before you move on

- Which of your pandas numbers did you reproduce in SQL, and did they agree to the rupee?
- Which query would you hand to finance, and could they read it without you in the room?
- What is the one index you would add if this ran weekly, and why not three?

## LOG 1 — SQL reconciliation (SQLite): `docs/decisions-log.md`, Reconciliation table

Rows in = rows kept + rows rejected for every table loaded, and the headline recomputed in SQL against pandas. Rows are appended once; re-running the cell does not duplicate them.

In [ ]:
STEP = "SQLite load (notebook 04)"
OWNER = "Rohit"
LOG = ROOT / "docs" / "decisions-log.md"
already = STEP in LOG.read_text(encoding="utf-8")

# SQL headline: H1 2026 tests from query 1 of 04_transformations.sql
sql_h1_2026 = int(results_04[0][1].set_index("half").loc["H1_2026", "tests"])
# pandas headline: same rule (exclude rescheduled, slot in H1 2026)
b = tables["bookings"]
mask = b["status_at_export"].fillna("").ne("rescheduled") & b["slot_at"].between(CONFIG["period_this"][0], "2026-06-30 23:59:59")
pd_h1_2026 = int(b.loc[mask, "n_tests"].sum())

rows = []
for name, df in tables.items():
    kept = int(pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {name}", con)["n"][0])
    rows.append((name, len(df), kept, len(df) - kept))
    headline = f"{kept:,} rows in the database"
    if name == "bookings":
        headline += f"; H1 2026 tests: SQL {sql_h1_2026:,} vs pandas {pd_h1_2026:,} ({'match' if sql_h1_2026 == pd_h1_2026 else 'DIFFERENT'})"
    if not already:
        logs.reconciliation(STEP, name, len(df), kept, len(df) - kept, headline, owner=OWNER)

recon = pd.DataFrame(rows, columns=["table", "rows_in", "rows_kept", "rows_rejected"])
recon["in_equals_kept_plus_rejected"] = recon["rows_in"] == recon["rows_kept"] + recon["rows_rejected"]
print(recon.to_string(index=False))
print("\nH1 2026 tests  SQL:", f"{sql_h1_2026:,}", " pandas:", f"{pd_h1_2026:,}")
print("Log written." if not already else "Already in the log; not appended again.")
assert recon["in_equals_kept_plus_rejected"].all() and recon["rows_rejected"].sum() == 0, "a table lost rows on load"

## LOG 2 — Cold run (SQLite): `docs/decisions-log.md`, Cold runs table

One row per full run: who, runtime of the SQL, the headline, and whether it matches the last run. Set `FRESH` and `SAME_AS_LAST` honestly before running.

In [ ]:
FRESH = False                 # True only if this ran in a fresh Codespace
SAME_AS_LAST = "first run"    # "yes" / "no" after comparing with the previous Cold runs row
WHY_DIFFERENT = ""

errors = [t for t, d in results_04 if "error" in d.columns]
assert not errors, f"these queries failed and were swallowed by run_sql: {errors}"

headline = f"H1 2026 tests {sql_h1_2026:,} (SQLite; pandas {pd_h1_2026:,})"
runtime = f"{sql_seconds:.0f}s for 04_transformations.sql on SQLite"
logs.cold_run(OWNER, FRESH, runtime, headline, SAME_AS_LAST, WHY_DIFFERENT)
print("Cold run logged:", runtime, "|", headline)